# Data Description
- Background: Unit 1.5分類依據的三個官方原始回應保存容器。不是都市計畫幾何、官方完整碼表或官方發布ZIP；外層ZIP由本地封裝，內部bytes為原回應。
- date received: 20261126（本次取得；UTC時間見manifest）
- Path to data file: 官方法規系統FL003962現行／20230804歷史匯出及都發局專有名詞頁，逐件原URL見manifest。
- Unit of observation: 一個原始HTTP回應（3件）＋一份manifest。
- Sample period: 112-08-04歷史法規、114-07-09修正現行條例及查閱時官方名詞；不認證分區圖資在2024–2025各時點效力。
- Known issues: 歷史匯出回應實為PDF，但保存member檔名以.html結尾；依Content-Type與PDF magic解析，不改bytes。沒有完整本機260碼對照，renderer不得冒充官方碼表，沒有重疊優先序。
- Definition for each variable: manifest欄位file=封裝member名、url=請求URL、retrieved_at_utc=取得時間、bytes=原回應大小、sha256=原回應hash、version_note=版本／限制註記、content_type=HTTP媒體型別、final_url=重導後URL、last_modified=HTTP欄位（可能缺）。
- Data path: `data/raw/unit15_zoning_classification_evidence.zip`
- Role: raw
- Verification status: 依下方乾淨kernel實際輸出，逐回應hash／bytes／版本及文字核對；不是GIS分析ready認證。
- SHA-256: `82effe1535fc7577d73accf81ee0663daa78944a11517b9e50c7bc2e197ba9b9`
- Next processing notebook: [Unit 1.5 validation](../code/taipei_urban_planning_geometry_validation.ipynb)
- Author: 詹明翰、謝子詮、曾翊涵（共同作者；不推定個別分工）
- Date first created: 20261126


In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,io,zipfile,sys,os,platform
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
def root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():return p
    raise FileNotFoundError('Open within the student project')
ROOT=root();TEMP=ROOT/'data/temp/unit15_validation';TEMP.mkdir(parents=True,exist_ok=True)
assert Path(sys.executable).resolve()==(ROOT/'.venv/Scripts/python.exe').resolve()
os.environ['PROJ_NETWORK']='OFF'
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def verify(bindings):
    rows=[]
    for relative,h in bindings.items():
        p=(ROOT/relative).resolve()
        assert p.is_relative_to((ROOT/'data').resolve()) and p.is_file(),relative
        assert sha(p)==h,'Input version changed: '+relative
        rows.append(dict(path=relative,bytes=p.stat().st_size,sha256=h))
    return pd.DataFrame(rows)
def show(title,frame,block=18):
    display(Markdown('**'+title+'**'))
    if len(frame)==0:display(Markdown('無符合紀錄。'))
    for start in range(0,len(frame),block):display(frame.iloc[start:start+block])
pd.set_option('display.max_colwidth',110)
pd.set_option('display.max_columns',22)
print({'python':platform.python_version(),'venv':True,'validation_date':'2026-11-26','network_in_kernel':False})


{'python': '3.12.2', 'venv': True, 'validation_date': '2026-11-26', 'network_in_kernel': False}


## Change for Your Own Data
固定唯一證據包，沒有網路下載或替換來源。


In [2]:
BINDINGS={'data/raw/unit15_zoning_classification_evidence.zip': '82effe1535fc7577d73accf81ee0663daa78944a11517b9e50c7bc2e197ba9b9'}
display(verify(BINDINGS))
import pymupdf
from bs4 import BeautifulSoup
with zipfile.ZipFile(ROOT/'data/raw/unit15_zoning_classification_evidence.zip') as z:
    manifest=pd.DataFrame(json.loads(z.read('source_manifest.json')))
    assert len(manifest)==3 and len(z.namelist())==4
    for r in manifest.to_dict('records'):
        b=z.read(r['file']);assert len(b)==r['bytes'] and hashlib.sha256(b).hexdigest()==r['sha256']
    historical=pymupdf.open(stream=z.read('taipei_zoning_regulation_20230804.html'),filetype='pdf')
    pdftext=''.join(p.get_text() for p in historical)
    current=BeautifulSoup(z.read('taipei_zoning_regulation_current.html'),'html.parser').get_text(' ',strip=True)
    terms=BeautifulSoup(z.read('udd_official_terms.html'),'html.parser').get_text(' ',strip=True)
assert '112 年 08 月 04 日' in pdftext and len(historical)==74
assert '第 3 條' in pdftext and all(w in pdftext and w in current for w in ['住宅區','商業區','工業區','文教區','保護區','特定專用區'])
assert '公共設施用地' in terms and '特定專用區' in terms
print({'verified_responses':len(manifest),'historical_pdf_pages':len(historical),'historical_version':'112-08-04','historical_member_actual_media':'application/pdf','classification_evidence_only':True})


,path,bytes,sha256
0,data/raw/unit15_zoning_classification_evidence.zip,285427,82effe1535fc7577d73accf81ee0663daa78944a11517b9e50c7bc2e197ba9b9


{'verified_responses': 3, 'historical_pdf_pages': 74, 'historical_version': '112-08-04', 'historical_member_actual_media': 'application/pdf', 'classification_evidence_only': True}


## Summary
### Sample


In [3]:
display(manifest.head(10))


,file,url,retrieved_at_utc,bytes,sha256,version_note,content_type,final_url,last_modified
0,taipei_zoning_regulation_current.html,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962,2026-11-26T07:18:08.450710+00:00,229418,7d5a52dfdd7d16895813561fc31e6ba33b20618fb257f0e89e87e6aae33bb5f8,114-07-09 修正；僅分類語義背景，不證明快照各地塊法定狀態,text/html; charset=utf-8,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962,None
1,taipei_zoning_regulation_20230804.html,https://www.laws.taipei.gov.tw/law/LawSearch/LawExport/FL003962?date=20230804&type=0,2026-11-26T07:18:09.854374+00:00,263760,7cc297bb8fec4768458f835628dacdc4637a481437bbb38bf9c2bcb409b1861f,112-08-04 歷史版；需由回應內容核對版本，非各細部計畫核定證明,application/pdf,https://laws.gov.taipei/law/LawSearch/LawExport/FL003962?date=20230804&type=0,None
2,udd_official_terms.html,https://udd.gov.taipei/PublicParticipation/noun-explanation,2026-11-26T07:18:09.977201+00:00,19105,e4ae8fd354cbd9fdf072cd817305f3ff475547501d06ce91c574bda94609bceb,都發局專有名詞：特定專用區、公共設施用地；網頁無完整分區碼表,text/html; charset=utf-8,https://udd.gov.taipei/PublicParticipation/noun-explanation,None


### Summary Stats


In [4]:
display(manifest[['bytes']].describe())
display(manifest.isna().sum().rename('missing').to_frame())
display(manifest.content_type.value_counts().rename_axis('HTTP media').reset_index(name='responses'))
print('三件回應的版本及類別語義存在；不擷取或重新分析圖資，不認證個別計畫，不聲稱證據補齊所有代碼。')


,bytes
count,3.00000
mean,170761.00000
std,132455.65293
min,19105.00000
25%,124261.50000
50%,229418.00000
75%,246589.00000
max,263760.00000


,missing
file,0
url,0
retrieved_at_utc,0
bytes,0
sha256,0
version_note,0
content_type,0
final_url,0
last_modified,3


,HTTP media,responses
0,text/html; charset=utf-8,2
1,application/pdf,1


三件回應的版本及類別語義存在；不擷取或重新分析圖資，不認證個別計畫，不聲稱證據補齊所有代碼。
